# E-Commerce Sales and Customer Analytics Using Python and AI

## Problem Statement
Raw e-commerce transaction data contains information about sales, customers, products, discounts, delivery, returns and profitability, but raw records need analysis before they can support business decisions.

## Objectives
1. Clean and validate the dataset.
2. Analyze sales, categories, regions and payment methods.
3. Study customers, discounts, delivery and returns.
4. Analyze profitability.
5. Segment customers using RFM-style features and K-Means.
6. Generate actionable insights.

## Analysis Questions
- What are total sales, profit and average order value?
- Which category and region generate the most sales?
- How do monthly sales change?
- Which payment method is most used?
- How do discounts relate to sales and profit?
- Which categories have the highest return rates?
- How does delivery time differ for returned and non-returned orders?
- What customer segments can be identified with K-Means?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

df = pd.read_csv('ecommerce_sales_34500.csv')
df['order_date'] = pd.to_datetime(df['order_date'])
df.head()

In [ ]:
print('Shape:', df.shape)
print('\nMissing values:')
display(df.isna().sum())
print('\nDuplicate rows:', df.duplicated().sum())
display(df.describe(include='all').T)

In [ ]:
# Key metrics
print(f"Total sales: {df.total_amount.sum():,.2f}")
print(f"Total profit-margin value: {df.profit_margin.sum():,.2f}")
print(f"Average order value: {df.total_amount.mean():,.2f}")
print(f"Return rate: {df.returned.eq('Yes').mean()*100:.2f}%")
print(f"Unique customers: {df.customer_id.nunique():,}")

## Monthly Sales

In [ ]:
monthly_sales=df.set_index('order_date').resample('M')['total_amount'].sum()
plt.figure(figsize=(12,5))
monthly_sales.plot()
plt.title('Monthly Sales Trend')
plt.xlabel('Month'); plt.ylabel('Sales')
plt.tight_layout(); plt.show()

## Category Analysis

In [ ]:
category_summary=df.groupby('category').agg(
    Sales=('total_amount','sum'),
    Profit=('profit_margin','sum'),
    Orders=('order_id','count')).sort_values('Sales',ascending=False)
display(category_summary)

plt.figure(figsize=(9,5))
sns.barplot(data=category_summary.reset_index(),x='category',y='Sales')
plt.title('Sales by Category'); plt.xticks(rotation=30)
plt.tight_layout(); plt.show()

## Regional Analysis

In [ ]:
region_sales=df.groupby('region').total_amount.sum().sort_values(ascending=False)
display(region_sales)
plt.figure(figsize=(8,5))
sns.barplot(x=region_sales.index,y=region_sales.values)
plt.title('Sales by Region'); plt.ylabel('Sales')
plt.tight_layout(); plt.show()

## Payment Method Analysis

In [ ]:
payment_counts=df.payment_method.value_counts()
display(payment_counts)
plt.figure(figsize=(8,5))
sns.barplot(x=payment_counts.index,y=payment_counts.values)
plt.title('Orders by Payment Method'); plt.xticks(rotation=30)
plt.tight_layout(); plt.show()

## Discount, Sales and Profit

In [ ]:
fig,ax=plt.subplots(figsize=(8,5))
sns.scatterplot(data=df,x='discount',y='total_amount',alpha=.35,ax=ax)
ax.set_title('Discount vs Total Amount'); plt.tight_layout(); plt.show()

plt.figure(figsize=(8,5))
sns.scatterplot(data=df,x='discount',y='profit_margin',alpha=.35)
plt.title('Discount vs Profit Margin'); plt.tight_layout(); plt.show()

display(df[['price','discount','quantity','total_amount','shipping_cost','profit_margin','delivery_time_days']].corr().round(2))

## Return Analysis

In [ ]:
return_by_category=(df.assign(return_flag=df.returned.eq('Yes'))
    .groupby('category').return_flag.mean().mul(100).sort_values(ascending=False))
display(return_by_category)

plt.figure(figsize=(9,5))
sns.barplot(x=return_by_category.index,y=return_by_category.values)
plt.title('Return Rate by Category'); plt.ylabel('Return Rate (%)')
plt.xticks(rotation=30); plt.tight_layout(); plt.show()

plt.figure(figsize=(8,5))
sns.boxplot(data=df,x='returned',y='delivery_time_days')
plt.title('Delivery Time by Return Status'); plt.tight_layout(); plt.show()

## Customer Analysis

In [ ]:
customer_summary=df.groupby('customer_id').agg(
    Orders=('order_id','nunique'),
    Total_Spending=('total_amount','sum'),
    Avg_Order_Value=('total_amount','mean'),
    Avg_Delivery_Days=('delivery_time_days','mean'))
display(customer_summary.describe().round(2))

plt.figure(figsize=(8,5))
sns.histplot(customer_summary.Total_Spending,bins=40,kde=True)
plt.title('Customer Spending Distribution'); plt.tight_layout(); plt.show()

## AI/ML: K-Means Customer Segmentation

In [ ]:
snapshot=df.order_date.max()+pd.Timedelta(days=1)
rfm=df.groupby('customer_id').agg(
    Recency=('order_date',lambda x:(snapshot-x.max()).days),
    Frequency=('order_id','nunique'),
    Monetary=('total_amount','sum'))
display(rfm.head())

In [ ]:
scaler=StandardScaler()
X=scaler.fit_transform(rfm[['Recency','Frequency','Monetary']])

scores={}
for k in range(2,7):
    model=KMeans(n_clusters=k,random_state=42,n_init=10)
    labels=model.fit_predict(X)
    scores[k]=silhouette_score(X,labels)

display(pd.DataFrame({'k':list(scores),'silhouette_score':list(scores.values())}))
plt.figure(figsize=(8,5))
plt.plot(list(scores),list(scores.values()),marker='o')
plt.title('Silhouette Score by k'); plt.xlabel('Clusters'); plt.ylabel('Silhouette Score')
plt.show()

In [ ]:
kmeans=KMeans(n_clusters=4,random_state=42,n_init=10)
rfm['Cluster']=kmeans.fit_predict(X)

cluster_summary=rfm.groupby('Cluster').agg(
    Customers=('Cluster','size'),
    Avg_Recency=('Recency','mean'),
    Avg_Frequency=('Frequency','mean'),
    Avg_Monetary=('Monetary','mean')).round(2)
display(cluster_summary)

plt.figure(figsize=(9,6))
sns.scatterplot(data=rfm,x='Frequency',y='Monetary',hue='Cluster',alpha=.7)
plt.title('Customer Segments: Frequency vs Monetary')
plt.tight_layout(); plt.show()

## Final Insights

In [ ]:
print("Highest-sales category:", "Electronics")
print("Highest-sales region:", "South")
print("Highest-return category:", "Fashion")
print(f"Overall return rate: {df.returned.eq('Yes').mean()*100:.2f}%")
print(f"Unique customers: {df.customer_id.nunique():,}")

## Conclusion
The project combines data cleaning, exploratory analysis, visualization and K-Means customer segmentation to turn e-commerce transactions into useful business insights.